In [1]:
!git clone -q https://github.com/poloclub/unitable.git /content/unitable
!pip install tokenizers apted lxml distance jsonlines huggingface_hub peft accelerate pandas tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.0 MB/s eta 0:00:00


In [2]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE = Path('/content/drive/MyDrive/Computer_Vision_Team8')
ZIP_PATH = BASE / 'data' / 'hierarchical_tables_v1.zip'

print(f'BASE exists: {BASE.exists()}')
print(f'ZIP exists : {ZIP_PATH.exists()}')

Mounted at /content/drive
BASE exists: True
ZIP exists : True


In [4]:
import subprocess
from pathlib import Path
EXTRACT_TO = Path('/content/data')
EXTRACT_TO.mkdir(parents=True, exist_ok=True)
existing = list(EXTRACT_TO.rglob('*.png')) + list(EXTRACT_TO.rglob('*.jpg'))
if existing:
    print(f'Already extracted — {len(existing)} images.')
else:
    print('Unzipping...')
    subprocess.run(['unzip', '-q', str(ZIP_PATH), '-d', str(EXTRACT_TO)], capture_output=True)
    imgs = list(EXTRACT_TO.rglob('*.png')) + list(EXTRACT_TO.rglob('*.jpg'))
    print(f'Done. {len(imgs)} images extracted.')

Unzipping...
Done. 32670 images extracted.


In [5]:
!python /content/run.py

[INFO] Using device: cuda:0
[INFO] Downloading unitable_large_structure.pt ...
unitable_large_structure.pt: 100% 497M/497M [00:04<00:00, 107MB/s]
[INFO] Downloading unitable_large_bbox.pt ...
unitable_large_bbox.pt: 100% 503M/503M [00:04<00:00, 119MB/s]
[INFO] Downloading unitable_large_content.pt ...
unitable_large_content.pt: 100% 527M/527M [00:05<00:00, 94.0MB/s]
[INFO] vocab_s size: 59
[INFO] vocab_b size: 891
[INFO] vocab_c size: 5363
[INFO] BOS IDs  — [html]=7, [bbox]=9, [cell]=8
[INFO] Bug 1 BOS token check PASSED ✅
[INFO] Loading structure model ...
  arch: d_model=768, nhead=12, enc_layers=12, dec_layers=4, ff_ratio=4, vocab=59, max_seq=784
/content/unitable/src/model/components.py:130: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=nlayer)
[INFO] Loading bbox model ...
  arch: d_model=768, nhead=12, enc_layers=12, dec_layers=4, ff_ratio=4,

## Experiement v2

In [ ]:
!python /content/run_experiment.py

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[INFO] Using device: cuda
UniTable Content Model 微調實驗 (run_experiment.py v2.1)

【配置摘要】
  SKIP_TRAINING: False
  TRAIN_EPOCHS: 3
  LEARNING_RATE: 5e-05
  BATCH_SIZE: 256
  USE_LORA: True
  LORA_RANK: 8, LORA_ALPHA: 16.0
  TRAIN_MAX_SAMPLES: 850
  VAL_MAX_SAMPLES: 150
  ORIGINAL_TEST_SAMPLES: 10
  FINETUNED_TEST_SAMPLES: 10
  COMPARE_WITH_ORIGINAL: True
  USE_COLAB: True

[Step 0] 載入 Vocab...
  vocab_c size: 5363

[Step 1] 開始訓練 (fine-tuning)...

[Step 1.1] 載入訓練數據...
[INFO] CSV 共有 1300 筆資料
[INFO] phase1 phase 有 1000 筆
[INFO] train split 有 850 筆
[INFO] 成功載入 850 筆
[INFO] CSV 共有 1300 筆資料
[INFO] phase1 phase 有 1000 筆
[INFO] val split 有 150 筆
[INFO] 成功載入 150 筆

[Step 1.2] 載入 BBox checkpoint...
[CHECKPOINT] Loaded BBox train: 3825 samples, 385607 cells
[CHECKPOINT] Loaded BBox val: 675 samples, 69038 cells

[Step 1.3] 建立 Dataset 和 DataLoader...
  Training cells: 70448
  V

In [ ]:
!python /content/debug_build_table_alignment.py

[INFO] Using device: cuda
Debug build_table_robust Alignment
[INFO] 輸出目錄: /content/drive/MyDrive/Computer_Vision_Team8/experiments/v2/outputs/debug_samples

[Step 1] 載入 Vocab...
  vocab_s size: 59
  vocab_b size: 891
  vocab_c size: 5363

[Step 2] 載入模型...
  載入 Structure model...
/content/unitable/src/model/components.py:130: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=nlayer)
  載入 BBox model...
  載入原始 Content model...
  載入 fine-tuned Content model: /content/drive/MyDrive/Computer_Vision_Team8/experiments/v2/checkpoints/content_best_merged.pt
  ✅ 所有模型載入完成

[Step 3] 讀取測試資料...
  Test split 共有 300 筆
  隨機選取 20 筆進行分析

[Step 4] 開始推論...

[1/20] fintabnet_059368
  GT cells: 13
  GT cells: 13, Orig: 13, FT: 13
  Saved to: /content/drive/MyDrive/Computer_Vision_Team8/experiments/v2/outputs/debug_samples/fintabnet_059368_*

[2/20] fintabnet_055146
  GT cells

In [ ]:
!python /content/run_benchmark_by_type.py

[INFO] Using device: cuda
UniTable Benchmark by Image Type

[Step 1] 載入 Vocab...
  vocab_s size: 59
  vocab_b size: 891
  vocab_c size: 5363

[Step 2] 載入模型...
  載入 Structure model...
/content/unitable/src/model/components.py:130: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=nlayer)
  載入 BBox model...
  載入 original Content model...
  ✅ original model 載入完成
  載入 fine-tuned Content model: /content/drive/MyDrive/Computer_Vision_Team8/experiments/v2/checkpoints/content_best_merged.pt
  ✅ 準備評估 2 個模型: ['original', 'finetuned']

[Step 3] 讀取測試資料...
  Test split 共有 300 筆
  Image types: ['wide_table', 'tall_table', 'low_contrast', 'normal_table', 'low_quality_blur']

[Step 4] 開始評估...

  Image type: wide_table (60 samples)

    評估 original model...
    [10/60] 完成 fintabnet_008415
    [20/60] 完成 fintabnet_000885
    [30/60] 完成 fintabnet_050632
    [40/60] 完成 fi

## Experiement v1

In [ ]:
!python /content/fine_tune_content_v2.py

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[INFO] Using device: cuda
UniTable Content Model Fine-tuning (v2 - CSV + LoRA)

[Step 1] 載入 Vocab...
[INFO] vocab_c size: 5363

[Step 2] 載入 BBox Model...
  arch: d_model=768, nhead=12, enc_layers=12, dec_layers=4, ff_ratio=4, vocab=891, max_seq=1024
/content/unitable/src/model/components.py:130: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=nlayer)

[Step 3] 載取資料... (IS_COLAB=True)
[INFO] CSV 路徑: /content/drive/MyDrive/Computer_Vision_Team8/data/training_fintabnet_pool_splits.csv
[INFO] Images 路徑: /content/data/data/processed/images
[INFO] 從 CSV 讀取資料: /content/drive/MyDrive/Computer_Vision_Team8/data/training_fintabnet_pool_splits.csv
[INFO] split = train, max_samples = 500
[INFO] CSV 共有 5500 筆資料
[INFO] 欄位: ['img_id', 'img_pa

In [ ]:
!python /content/fine_tune_content_v2_fixed.py

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[INFO] Using device: cuda
UniTable Content Model Fine-tuning (v2.2 Pipeline Run)

[Step 1] Loading Vocab...
[INFO] vocab_c size: 5363

[Step 2] Loading BBox Model...
  arch: d_model=768, nhead=12, enc_layers=12, dec_layers=4, ff_ratio=4, vocab=891, max_seq=1024
/content/unitable/src/model/components.py:130: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=nlayer)

[Step 3] Loading data... (IS_COLAB=True)
[INFO] Loading data from CSV: /content/drive/MyDrive/Computer_Vision_Team8/data/training_fintabnet_pool_splits.csv
[INFO] split = train, max_samples = 100
[INFO] CSV has 5500 records
[INFO] train split has 3825 records
[INFO] Loaded 100 records
[INFO] Loading data from CSV: /content/drive/MyDrive/Computer_Vision_Team8/data/train

In [ ]:
!python /content/verify_merged_checkpoint.py

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[INFO] Using device: cuda

[Step 1] 載入原始 Content Model...
  路徑: /content/unitable/experiments/unitable_weights/unitable_large_content.pt
  arch: d_model=768, nhead=12, enc_layers=12, dec_layers=4, ff_ratio=4, vocab=5363, max_seq=200
/content/unitable/src/model/components.py:130: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=nlayer)
  ✅ 原始模型建立完成

[Step 2] 載入 LoRA checkpoint...
  路徑: /content/drive/MyDrive/Computer_Vision_Team8/models/ryan/outputs/checkpoints/content_best.pt
  LoRA checkpoint 共有 288 個 key
  [INFO] 確認這是 LoRA checkpoint

  [DEBUG] Checkpoint 中的前 10 個 key:
    [0] base_model.model.backbone.conv_proj.weight
    [1] base_model.model.backbone.conv_proj.bias
    [2] base_model.model.encoder.encoder.layers.0.self_attn.

In [ ]:
!python /content/run_comparison.py

[INFO] Using device: cuda

[Step 1] 載入 Vocab...
[INFO] vocab_s size: 59
[INFO] vocab_b size: 891
[INFO] vocab_c size: 5363

[Step 2] 載入 Structure 和 BBox 模型...
  arch: d_model=768, nhead=12, enc_layers=12, dec_layers=4, ff_ratio=4, vocab=59, max_seq=784
/content/unitable/src/model/components.py:130: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=nlayer)
  arch: d_model=768, nhead=12, enc_layers=12, dec_layers=4, ff_ratio=4, vocab=891, max_seq=1024

[Step 3] 載入 Content 模型...
  載入原始 Content Model...
  arch: d_model=768, nhead=12, enc_layers=12, dec_layers=4, ff_ratio=4, vocab=5363, max_seq=200
  ✅ 原始模型載入完成
  載入 merge 後微調 Content Model: /content/drive/MyDrive/Computer_Vision_Team8/models/ryan/outputs/checkpoints/content_best_merged.pt
  arch: d_model=768, nhead=12, enc_layers=12, dec_layers=4, ff_ratio=4, vocab=5363, max_seq=200
  ✅ Merge 後微調模型載入完成

[St